# 不明を含む高中低データ：pgmpyによる検証
一つのNotebookに全コードを収録。Python (bn)で上から実行します。実務への合格判定ではなく、指定された10観点の検証と未確認事項を分けます。

`df` がなければ人工データを生成します。不明は未観測としてNaNへ変換し、高中低の第四カテゴリにはしません。不明そのものに意味がある業務ではこの仮定を見直してください。
主解析は学習データ内の最頻値補完＋HC/BIC＋BDeuパラメータESS=10と**事前固定**します。比較の勝者へ最終評価を合わせ直しません。

In [1]:
from pathlib import Path
import json, importlib.metadata as metadata
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
OUT=Path('results'); OUT.mkdir(exist_ok=True)
required={'pgmpy':'1.1.2','pandas':'2.3.3','numpy':'2.3.5','scikit-learn':'1.9.0'}
versions={p:metadata.version(p) for p in required}
assert versions==required, versions
print(versions)
def save(name, table):
    table.to_csv(OUT/(name+'.csv'),index=False,encoding='utf-8-sig')
    return table

{'pgmpy': '1.1.2', 'pandas': '2.3.3', 'numpy': '2.3.5', 'scikit-learn': '1.9.0'}


## 共通関数
構造・確率推定・検証を短い関数に分けます。主な処理のループは後段にあります。BDeuで全3状態を保持し、未観測の親条件を最尤推定で識別できたようには扱いません。

In [2]:
from itertools import combinations, product
import logging
import warnings
import numpy as np
import pandas as pd
import networkx as nx
from scipy.special import logsumexp
from scipy.stats import beta, chi2_contingency
from sklearn.model_selection import KFold
from pgmpy.base import DAG
from pgmpy.causal_discovery import HillClimbSearch, PC
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteMLE, DiscreteBayesianEstimator
from pgmpy.structure_score import BIC, BDeu
from pgmpy.inference import VariableElimination
STATES=["低","中","高"]
logging.getLogger("pgmpy").setLevel(logging.ERROR)

/opt/anaconda3/envs/bn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/bn/lib/python3.11/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [3]:
def make_sample(n=1200, seed=42):
    """Artificial business-like associations; generation graph is NOT a validation target."""
    rng = np.random.default_rng(seed)
    quality = rng.choice([1, 2, 3], n, p=[.25, .45, .30])
    value = rng.choice([1, 2, 3], n, p=[.30, .40, .30])
    speed = np.where(rng.random(n) < .65, quality, rng.integers(1, 4, n))
    discretize = lambda v: np.digitize(v, [1.6, 2.4]) + 1
    satisfaction = discretize(.5*quality + .35*value + .15*speed + rng.normal(0, .45, n))
    renewal = discretize(satisfaction + rng.normal(0, .6, n))
    arrays = [quality, value, speed, satisfaction, renewal]
    names = ['サービス品質', '価格納得感', '対応速度', '満足度', '継続意向']
    result = pd.DataFrame({c: np.array(STATES)[a-1] for c, a in zip(names, arrays)})
    result.attrs['data_source'] = f'Python生成の人工データ（n={n}, seed={seed}）'
    return result

def state_names(data):
    return {c: STATES.copy() for c in data.columns}

def learn_structure(data, algorithm, max_parents=2, structure_ess=10):
    if algorithm == 'Independent':
        graph = DAG()
        graph.add_nodes_from(data.columns)
        return graph
    if algorithm in ['HC_BIC', 'HC_BDeu']:
        score = (BIC(data, state_names=state_names(data)) if algorithm == 'HC_BIC'
                 else BDeu(data, equivalent_sample_size=structure_ess, state_names=state_names(data)))
        est = HillClimbSearch(scoring_method=score, max_indegree=max_parents,
                             return_type='dag', show_progress=False, max_iter=10000)
    elif algorithm == 'PC':
        # PC's conditioning-set cap is NOT a maximum-parent constraint.
        est = PC(variant='stable', ci_test='chi_square', significance_level=.01,
                 max_cond_vars=2, return_type='dag', show_progress=False, n_jobs=1)
    else:
        raise ValueError(algorithm)
    graph = est.fit(data).causal_graph_
    graph.add_nodes_from(data.columns)
    return graph

def support_table(graph, data):
    records = []
    for node in graph.nodes():
        parents = sorted(graph.predecessors(node))
        if 3**len(parents) * 3 > 10000:
            raise ValueError('CPT too large: reduce parents before fitting.')
        for values in product(STATES, repeat=len(parents)):
            mask = np.ones(len(data), dtype=bool)
            for col, value in zip(parents, values):
                mask &= data[col].to_numpy() == value
            counts = data.loc[mask, node].value_counts().reindex(STATES, fill_value=0)
            records.append(dict(node=node, parents=tuple(parents), context=tuple(values),
                                n=int(mask.sum()), counts=counts.to_numpy(dtype=int)))
    return records

def fit_parameters(graph, data, ess):
    """ess=None: MLE, positive ess: BDeu posterior means."""
    support = support_table(graph, data)
    # pgmpy MLE fills entirely unseen parent contexts with uniform probabilities.
    # Flag those as unidentifiable instead of presenting this fallback as evidence.
    if ess is None and any(r['n'] == 0 for r in support):
        raise ValueError('unidentified_MLE: unobserved parent configuration (uniform fallback rejected)')
    model = DiscreteBayesianNetwork(graph.edges())
    model.add_nodes_from(graph.nodes())  # Preserve isolated variables.
    estimator = (DiscreteMLE(state_names=state_names(data)) if ess is None else
                 DiscreteBayesianEstimator(state_names=state_names(data), prior_type='BDeu',
                                           equivalent_sample_size=float(ess)))
    cpds = estimator.fit(model, data).parameters_
    model.add_cpds(*cpds)
    assert model.check_model() and len(cpds) == data.shape[1]
    for cpd in cpds:
        assert cpd.state_names[cpd.variable] == STATES
        assert np.allclose(cpd.get_values().sum(axis=0), 1)
    return model

In [4]:
def row_nll(model, data):
    """Exact joint negative log probability, natural logarithm. No clipping of zero."""
    values = np.zeros(len(data))
    for cpd in model.get_cpds():
        idx = tuple(data[v].map({s: i for i, s in enumerate(cpd.state_names[v])})
                    .to_numpy(dtype=int) for v in cpd.variables)
        probs = np.asarray(cpd.values)[idx]
        with np.errstate(divide='ignore'):
            values -= np.log(probs)
    return values

def score_structures(data, algorithms, max_parents=2):
    records, graphs = [], {}
    for algorithm in algorithms:
        graph = learn_structure(data, algorithm, max_parents)
        k = sum(2 * 3**len(list(graph.predecessors(v))) for v in graph.nodes())
        bic = BIC(data, state_names=state_names(data)).score(graph)
        records.append(dict(algorithm=algorithm, edges=graph.number_of_edges(), parameters=k,
                            BIC=bic, BDeu_ESS10=BDeu(data, equivalent_sample_size=10,
                            state_names=state_names(data)).score(graph)))
        graphs[algorithm] = graph
    return pd.DataFrame(records), graphs

def cpt_details(graph, model, train, ess):
    rows = []
    for r in support_table(graph, train):
        q = 3**len(r['parents'])
        alpha = ess/(3*q) if ess is not None else 0
        for index, state in enumerate(STATES):
            count = r['counts'][index]
            assignment = dict(zip(r['parents'], r['context'])) | {r['node']: state}
            probability = model.get_cpds(r['node']).get_value(**assignment)
            expected = (count + alpha)/(r['n'] + 3*alpha)
            assert np.isclose(probability, expected)
            low, high = (beta.ppf([.025, .975], count+alpha, r['n']-count+2*alpha)
                         if ess is not None else (np.nan, np.nan))
            rows.append(dict(node=r['node'], parents=' / '.join(r['parents']) or 'なし',
                             context=' / '.join(r['context']) or '全体', state=state,
                             n_context=r['n'], count=int(count), probability=probability,
                             lower95=low, upper95=high, interval_width=high-low,
                             prior_fraction=(3*alpha)/(r['n']+3*alpha),
                             no_data=r['n']==0))
    return pd.DataFrame(rows)

def wilson(success, n, z=1.95996398454):
    if n == 0:
        return np.nan, np.nan
    p = success/n
    center = (p+z*z/(2*n))/(1+z*z/n)
    half = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)
    return center-half, center+half

def run_bootstrap(data, algorithms, seeds, repetitions=300):
    """Record every pair, even absent edges, and failures with their denominators."""
    pairs = list(combinations(data.columns, 2))
    rows, errors = [], []
    for algorithm in algorithms:
        for seed in seeds:
            rng = np.random.default_rng(seed)
            for repetition in range(repetitions):
                sample = data.iloc[rng.integers(0, len(data), len(data))]
                try:
                    graph = learn_structure(sample, algorithm)
                    cpdag = graph.to_pdag()
                    for a, b in pairs:
                        present = graph.has_edge(a,b) or graph.has_edge(b,a)
                        directed_ab = (a,b) in cpdag.directed_edges
                        directed_ba = (b,a) in cpdag.directed_edges
                        rows.append(dict(algorithm=algorithm, seed=seed, repetition=repetition,
                                         a=a, b=b, present=present, a_to_b=directed_ab,
                                         b_to_a=directed_ba,
                                         unresolved=present and not directed_ab and not directed_ba))
                except Exception as exc:
                    errors.append(dict(algorithm=algorithm, seed=seed, repetition=repetition,
                                       error=str(exc)))
            print(f'Bootstrap {algorithm} seed={seed}: {repetitions} attempts', flush=True)
    detail = pd.DataFrame(rows)
    summary = []
    for (algorithm, a, b), g in detail.groupby(['algorithm', 'a', 'b'], sort=False):
        n, hits = len(g), int(g.present.sum())
        low, high = wilson(hits,n)
        seed_rates = g.groupby('seed').present.mean()
        summary.append(dict(algorithm=algorithm, a=a, b=b, successful=n,
                            planned=len(seeds)*repetitions, strength=hits/n,
                            failed=len(seeds)*repetitions-n,
                            failure_bound_lower=hits/(len(seeds)*repetitions),
                            failure_bound_upper=(hits+len(seeds)*repetitions-n)/(len(seeds)*repetitions),
                            mc_lower95=low, mc_upper95=high,
                            seed_min=seed_rates.min(), seed_max=seed_rates.max(),
                            direction_ab=g.a_to_b.sum()/hits if hits else np.nan,
                            direction_ba=g.b_to_a.sum()/hits if hits else np.nan,
                            direction_unresolved=g.unresolved.sum()/hits if hits else np.nan))
    return detail, pd.DataFrame(summary), pd.DataFrame(errors, columns=['algorithm','seed','repetition','error'])

In [5]:
def holm(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    order = np.argsort(pvalues)
    adjusted = np.minimum(1, np.maximum.accumulate(pvalues[order]*(len(pvalues)-np.arange(len(pvalues)))))
    result = np.empty(len(pvalues)); result[order] = adjusted
    return result

def conditional_mi(x, y, strata):
    result = 0.
    for indices in strata:
        table = np.bincount(x[indices]*3+y[indices], minlength=9).reshape(3,3)
        n = table.sum()
        expected = np.outer(table.sum(axis=1), table.sum(axis=0))/n
        nonzero = table > 0
        result += np.sum(table[nonzero]*np.log(table[nonzero]/expected[nonzero]))
    return result/len(x)

def graph_test_plan(graph):
    """Lock hypotheses using development data only, before examining holdout."""
    plan = []
    seen = set()
    for node in graph.nodes():
        parents = set(graph.predecessors(node))
        for other in graph.nodes():
            if other == node or other in nx.descendants(graph, node):
                continue
            kind = 'edge' if other in parents else 'implied_independence'
            given = tuple(sorted(parents-{other}))
            key = (tuple(sorted([node, other])), given)
            if key in seen:
                continue
            seen.add(key)
            plan.append(dict(kind=kind, x=node, y=other, given=given))
    return plan

def conditional_tests(test, plan, permutations=1999, seed=707):
    rng = np.random.default_rng(seed)
    rows = []
    for hypothesis in plan:
        x = test[hypothesis['x']].cat.codes.to_numpy()
        y = test[hypothesis['y']].cat.codes.to_numpy()
        z = list(hypothesis['given'])
        strata = ([np.asarray(v) for v in test.reset_index(drop=True).groupby(z, observed=True).indices.values()]
                  if z else [np.arange(len(test))])
        testable = any(len(np.unique(x[ix]))>1 and len(np.unique(y[ix]))>1 for ix in strata)
        observed = conditional_mi(x,y,strata)
        extreme = 0
        if testable:
            for _ in range(permutations):
                yp = y.copy()
                for ix in strata:
                    yp[ix] = rng.permutation(yp[ix])
                extreme += conditional_mi(x,yp,strata) >= observed-1e-14
        rows.append(dict(kind=hypothesis['kind'], x=hypothesis['x'], y=hypothesis['y'],
                         given=' / '.join(z) or 'なし', conditional_MI=observed,
                         p=(extreme+1)/(permutations+1) if testable else np.nan,
                         permutations=permutations, testable=testable,
                         n=len(test), min_stratum_n=min(map(len,strata))))
    result = pd.DataFrame(rows)
    result['p_holm'] = np.nan
    valid = result.p.notna()
    result.loc[valid, 'p_holm'] = holm(result.loc[valid,'p'])
    return result

def paired_loss_interval(model, baseline, test, repetitions=2000, seed=818):
    # Positive improvement means the selected model has smaller loss.
    delta = row_nll(baseline,test)-row_nll(model,test)
    if not np.isfinite(delta).all():
        return dict(improvement=np.nan,lower95=np.nan,upper95=np.nan,status='nonfinite_loss')
    rng=np.random.default_rng(seed)
    means=[delta[rng.integers(0,len(delta),len(delta))].mean() for _ in range(repetitions)]
    lo,hi=np.quantile(means,[.025,.975])
    return dict(improvement=delta.mean(),lower95=lo,upper95=hi,status='ok')

## 1. dfの生成と未知データの隔離
5項目×1,500行を生成し、約6%のセルを不明にします。満足度が低い行の価格納得感では不明率をさらに上げます（欠損が完全にランダムではない例）。生成前の完全データは分析や補完には使いません。
20%を最終評価として隔離します。評価は全項目を観測できた行に共通化するため、結果の対象は**完全観測の部分集団**です。欠損のある全顧客への性能を保証しません。

In [6]:
if 'df' not in globals():
    df=make_sample(1500,42)
    rng=np.random.default_rng(43)
    missing=rng.random(df.shape)<.06
    missing[:,1] |= (df['満足度'].to_numpy()=='低') & (rng.random(len(df))<.15)
    df=df.mask(missing,'不明')
assert isinstance(df,pd.DataFrame) and df.columns.is_unique and len(df)>=50
raw=df.copy(deep=True).replace('不明',np.nan)
for col in raw:
    assert raw[col].dropna().isin(STATES).all(), f'{col}: 不正な値'
    assert raw[col].nunique()>=2, f'{col}: 状態が不足'
raw=raw.reset_index(drop=True)
display(save('missing_rates',raw.isna().mean().rename('missing_rate').reset_index()))
save('input_df',df)
train_idx,test_idx=train_test_split(np.arange(len(raw)),test_size=.2,random_state=42)
dev=raw.iloc[train_idx].reset_index(drop=True)
test_raw=raw.iloc[test_idx].reset_index(drop=True)
save('split',pd.DataFrame({'row':np.arange(len(raw)),'split':np.where(np.isin(np.arange(len(raw)),test_idx),'test','dev')}))

def categorical(data):
    return data.apply(lambda col: pd.Series(pd.Categorical(col,categories=STATES),index=col.index))

def preprocess_train(data, method, seed=1):
    if method=='complete_case':
        result=data.dropna().copy()
    else:
        result=data.copy();rng=np.random.default_rng(seed)
        for col in result:
            observed=result[col].dropna()
            if len(observed)==0: raise ValueError(f'{col}: 全欠損')
            mask=result[col].isna()
            if method=='mode':
                result.loc[mask,col]=observed.mode().iloc[0]
            elif method=='random_marginal':
                frequencies=observed.value_counts(normalize=True).reindex(STATES,fill_value=0)
                result.loc[mask,col]=rng.choice(STATES,int(mask.sum()),p=frequencies.to_numpy())
            else: raise ValueError(method)
    if len(result)<20: raise ValueError('学習行不足')
    return categorical(result).reset_index(drop=True)

train=preprocess_train(dev,'mode')
test=categorical(test_raw.dropna()).reset_index(drop=True)
assert len(test)>=20
print('開発行:',len(dev),'最終評価の完全観測行:',len(test),'/',len(test_raw))

,index,missing_rate
0,サービス品質,0.055333
1,価格納得感,0.106000
2,対応速度,0.062000
3,満足度,0.073333
4,継続意向,0.067333


開発行: 1200 最終評価の完全観測行: 209 / 300


## 2. アルゴリズム・スコア・対数尤度・パラメータ数
HC/BIC、HC/BDeu、PC、辺なしの基準モデルを比較します。HC2設定は同じ探索法、PCは独立性検定に基づく別手法です。
BICとBDeuは同一標本内で**高い方が良い**。異なる種類のスコア同士の数値を直接比べません。BICはlog L − k log(n)/2という符号です。
`logLik_fitted` は平滑化したモデルの対数尤度、`logLik_MLE_from_BIC` は構造スコアから復元した最尤対数尤度で、同じものではありません。
パラメータ数はΣ(3−1)×3^親数です。多いこと自体は優劣ではありません。

In [7]:
algorithms=['HC_BIC','HC_BDeu','PC','Independent']
scores,graphs=score_structures(train,algorithms)
models={name:fit_parameters(graph,train,10) for name,graph in graphs.items()}
scores['logLik_fitted']=[-row_nll(models[name],train).sum() for name in scores.algorithm]
scores['logLik_MLE_from_BIC']=scores.BIC+.5*scores.parameters*np.log(len(train))
display(save('scores',scores))
model=models['HC_BIC'];graph=graphs['HC_BIC']

,algorithm,edges,parameters,BIC,BDeu_ESS10,logLik_fitted,logLik_MLE_from_BIC
0,HC_BIC,4,34,-5496.150435,-5465.901949,-5376.639609,-5375.619129
1,HC_BDeu,4,34,-5496.150435,-5465.901949,-5376.639609,-5375.619129
2,PC,4,34,-5496.150435,-5465.901949,-5376.639609,-5375.619129
3,Independent,0,10,-6433.917451,-6424.098295,-6398.479746,-6398.467067


## 3. 交差検証と欠損処理の比較
3反復×5fold。分割→学習側だけで欠損処理→構造再学習→確率推定→検証の順です。
完全ケース除去、最頻値補完、周辺分布からの確率的補完を比較します。最後は**単一補完**であり、多重代入による不確実性評価ではありません。周辺補完は項目間の関係を弱める可能性があります。
全候補を同じ完全観測検証行で比較します。joint NLLは低い方が良く、失敗を除いた都合の良い平均は作りません。foldは独立ではないため平均のt検定はしません。

In [8]:
cv_rows=[]
for repeat in range(3):
    for fold,(itr,iva) in enumerate(KFold(5,shuffle=True,random_state=100+repeat).split(dev)):
        valid=categorical(dev.iloc[iva].dropna()).reset_index(drop=True)
        for method in ['complete_case','mode','random_marginal']:
            fitted_data=preprocess_train(dev.iloc[itr],method,seed=1000+repeat*5+fold)
            for algorithm in algorithms:
                try:
                    g=learn_structure(fitted_data,algorithm)
                    fit=fit_parameters(g,fitted_data,10)
                    loss=row_nll(fit,valid).mean()
                    status='ok' if np.isfinite(loss) else 'nonfinite'
                    error=''
                except Exception as exc:
                    loss=np.nan;status='error';error=str(exc)
                cv_rows.append(dict(repeat=repeat,fold=fold,method=method,algorithm=algorithm,
                    n_train=len(fitted_data),n_test=len(valid),nll=loss,status=status,error=error))
cv=save('cv_detail',pd.DataFrame(cv_rows))
summary=[]
for (method,algorithm),g in cv.groupby(['method','algorithm']):
    summary.append(dict(method=method,algorithm=algorithm,failures=int((g.status!='ok').sum()),
        nll=np.average(g.nll,weights=g.n_test) if (g.status=='ok').all() else np.nan))
cv_summary=save('cv_summary',pd.DataFrame(summary));display(cv_summary)
holdout=save('holdout',pd.DataFrame([dict(model_nll=row_nll(model,test).mean(),
    baseline_nll=row_nll(models['Independent'],test).mean(),**paired_loss_interval(model,models['Independent'],test))]))
display(holdout)

,method,algorithm,failures,nll
0,complete_case,HC_BDeu,0,4.362831
1,complete_case,HC_BIC,0,4.385544
2,complete_case,Independent,0,5.426487
3,complete_case,PC,0,4.366331
4,mode,HC_BDeu,0,4.416179
5,mode,HC_BIC,0,4.441026
6,mode,Independent,0,5.453244
7,mode,PC,0,4.413119
8,random_marginal,HC_BDeu,0,4.417947
9,random_marginal,HC_BIC,0,4.431820


,model_nll,baseline_nll,improvement,lower95,upper95,status
0,4.353011,5.429501,1.07649,0.9235,1.243346,ok


## 4. ブートストラップ：存在と方向
各手法200回×3seed。**不明を含む元の開発行を再抽出して毎回最頻値補完をやり直す**ことで、主解析手続き全体を繰り返します。
CPDAGで方向確定／逆方向確定／未確定を区別します。頻度は真の辺が存在する事後確率ではありません。区間は成功回に条件づけたMonte Carlo計算誤差のみ。
600回の最悪ケースで近似95%半幅は約4ポイントです。2ポイントを目指すなら約2,401回／手法が必要です。失敗の影響範囲も保存します。

In [9]:
# 元の関数と同じ集計を用い、学習呼出し直前に補完する専用ラッパーを定義します。
def bootstrap_with_missing(data, algorithms, seeds, repetitions=300):
    """Record every pair, even absent edges, and failures with their denominators."""
    pairs = list(combinations(data.columns, 2))
    rows, errors = [], []
    for algorithm in algorithms:
        for seed in seeds:
            rng = np.random.default_rng(seed)
            for repetition in range(repetitions):
                sample = data.iloc[rng.integers(0, len(data), len(data))]
                try:
                    sample = preprocess_train(sample, "mode")
                    graph = learn_structure(sample, algorithm)
                    cpdag = graph.to_pdag()
                    for a, b in pairs:
                        present = graph.has_edge(a,b) or graph.has_edge(b,a)
                        directed_ab = (a,b) in cpdag.directed_edges
                        directed_ba = (b,a) in cpdag.directed_edges
                        rows.append(dict(algorithm=algorithm, seed=seed, repetition=repetition,
                                         a=a, b=b, present=present, a_to_b=directed_ab,
                                         b_to_a=directed_ba,
                                         unresolved=present and not directed_ab and not directed_ba))
                except Exception as exc:
                    errors.append(dict(algorithm=algorithm, seed=seed, repetition=repetition,
                                       error=str(exc)))
            print(f'Bootstrap {algorithm} seed={seed}: {repetitions} attempts', flush=True)
    detail = pd.DataFrame(rows)
    summary = []
    for (algorithm, a, b), g in detail.groupby(['algorithm', 'a', 'b'], sort=False):
        n, hits = len(g), int(g.present.sum())
        low, high = wilson(hits,n)
        seed_rates = g.groupby('seed').present.mean()
        summary.append(dict(algorithm=algorithm, a=a, b=b, successful=n,
                            planned=len(seeds)*repetitions, strength=hits/n,
                            failed=len(seeds)*repetitions-n,
                            failure_bound_lower=hits/(len(seeds)*repetitions),
                            failure_bound_upper=(hits+len(seeds)*repetitions-n)/(len(seeds)*repetitions),
                            mc_lower95=low, mc_upper95=high,
                            seed_min=seed_rates.min(), seed_max=seed_rates.max(),
                            direction_ab=g.a_to_b.sum()/hits if hits else np.nan,
                            direction_ba=g.b_to_a.sum()/hits if hits else np.nan,
                            direction_unresolved=g.unresolved.sum()/hits if hits else np.nan))
    return detail, pd.DataFrame(summary), pd.DataFrame(errors, columns=['algorithm','seed','repetition','error'])
boot_detail,edges,boot_errors=bootstrap_with_missing(dev,['HC_BIC','HC_BDeu','PC'],[101,202,303],200)
save('bootstrap_detail',boot_detail);save('edge_stability',edges);save('bootstrap_errors',boot_errors)
display(edges)
print('学習失敗:',len(boot_errors))

Bootstrap HC_BIC seed=101: 200 attempts


Bootstrap HC_BIC seed=202: 200 attempts


Bootstrap HC_BIC seed=303: 200 attempts


Bootstrap HC_BDeu seed=101: 200 attempts


Bootstrap HC_BDeu seed=202: 200 attempts


Bootstrap HC_BDeu seed=303: 200 attempts


Bootstrap PC seed=101: 200 attempts


Bootstrap PC seed=202: 200 attempts


Bootstrap PC seed=303: 200 attempts


,algorithm,a,b,successful,planned,strength,failed,failure_bound_lower,failure_bound_upper,mc_lower95,mc_upper95,seed_min,seed_max,direction_ab,direction_ba,direction_unresolved
0,HC_BIC,サービス品質,価格納得感,600,600,0.135000,0,0.135000,0.135000,0.109967,0.164677,0.115,0.145,0.000000,0.000000,1.000000
1,HC_BIC,サービス品質,対応速度,600,600,1.000000,0,1.000000,1.000000,0.993638,1.000000,1.000,1.000,0.000000,0.000000,1.000000
2,HC_BIC,サービス品質,満足度,600,600,1.000000,0,1.000000,1.000000,0.993638,1.000000,1.000,1.000,0.548333,0.000000,0.451667
3,HC_BIC,サービス品質,継続意向,600,600,0.018333,0,0.018333,0.018333,0.010267,0.032528,0.015,0.020,0.000000,0.000000,1.000000
4,HC_BIC,価格納得感,対応速度,600,600,0.018333,0,0.018333,0.018333,0.010267,0.032528,0.015,0.020,0.000000,0.818182,0.181818
5,HC_BIC,価格納得感,満足度,600,600,0.998333,0,0.998333,0.998333,0.990620,0.999706,0.995,1.000,0.549249,0.015025,0.435726
6,HC_BIC,価格納得感,継続意向,600,600,0.001667,0,0.001667,0.001667,0.000294,0.009380,0.000,0.005,0.000000,0.000000,1.000000
7,HC_BIC,対応速度,満足度,600,600,0.056667,0,0.056667,0.056667,0.040832,0.078142,0.045,0.075,0.000000,0.000000,1.000000
8,HC_BIC,対応速度,継続意向,600,600,0.000000,0,0.000000,0.000000,0.000000,0.006362,0.000,0.000,NaN,NaN,NaN
9,HC_BIC,満足度,継続意向,600,600,1.000000,0,1.000000,1.000000,0.993638,1.000000,1.000,1.000,0.548333,0.000000,0.451667


学習失敗: 0


## 5. マルコフ同値性と構造距離
`to_pdag()`で同値類を表示します。未確定辺は観測上の独立性だけでは向きが一意になりません。
SHDはここでは「追加・削除・反転を各1」と数え、骨格の差も別に示します。Rとの実装・定義差を明示するため独自の単純な距離関数です。真の構造との距離ではありません。

In [10]:
def graph_distance(a,b):
    shd=skeleton=0
    for x,y in combinations(a.nodes(),2):
        def state(g): return 1 if g.has_edge(x,y) else (-1 if g.has_edge(y,x) else 0)
        u,v=state(a),state(b)
        shd+=u!=v;skeleton+=(u==0)!=(v==0)
    return shd,skeleton
same=[];equivalence=[]
for name,g in graphs.items():
    cp=g.to_pdag()
    equivalence.append(dict(algorithm=name,compelled=str(sorted(cp.directed_edges)),
        reversible=str(sorted({tuple(sorted(e)) for e in cp.undirected_edges}))))
for a,b in combinations(graphs,2):
    shd,skeleton=graph_distance(graphs[a],graphs[b])
    same.append(dict(a=a,b=b,SHD=shd,skeleton_difference=skeleton,
        markov_equivalent=graphs[a].is_iequivalent(graphs[b])))
display(save('equivalence',pd.DataFrame(equivalence)),save('structure_comparison',pd.DataFrame(same)))

,algorithm,compelled,reversible
0,HC_BIC,"[('サービス品質', '満足度'), ('価格納得感', '満足度'), ('満足度', ...","[('サービス品質', '対応速度')]"
1,HC_BDeu,"[('サービス品質', '満足度'), ('価格納得感', '満足度'), ('満足度', ...","[('サービス品質', '対応速度')]"
2,PC,"[('サービス品質', '満足度'), ('価格納得感', '満足度'), ('満足度', ...","[('サービス品質', '対応速度')]"
3,Independent,[],[]


,a,b,SHD,skeleton_difference,markov_equivalent
0,HC_BIC,HC_BDeu,0,0,True
1,HC_BIC,PC,0,0,True
2,HC_BIC,Independent,4,4,False
3,HC_BDeu,PC,0,0,True
4,HC_BDeu,Independent,4,4,False
5,PC,Independent,4,4,False


## 6. 辺のスコア寄与とベイズファクター
各選択辺を一つ除いたグラフと比較します。ΔBIC=元−削除後、log BF=BDeu(元)−BDeu(削除後)。正は辺を残す側を支持します。
さらに各ペアについて、そのペア以外の構造を固定し「なし／a→b／b→a」の合法なDAGだけを比較します。BDeuと3状態への等事前確率から条件付きモデル重みを計算します。
これは**固定された残りの構造の下の局所比較**で、全DAGを積分した辺の事後確率ではありません。bnlearnのbf.strengthの完全再実装とは呼びません。数値発散を避けlog BFで出力します。

In [11]:
contributions=[];bf_rows=[]
bic=BIC(train,state_names=state_names(train));bdeu=BDeu(train,equivalent_sample_size=10,state_names=state_names(train))
for a,b in graph.edges():
    removed=graph.copy();removed.remove_edge(a,b)
    contributions.append(dict(a=a,b=b,delta_BIC=bic.score(graph)-bic.score(removed),
        log_BF=bdeu.score(graph)-bdeu.score(removed)))
for a,b in combinations(graph.nodes(),2):
    base=graph.copy()
    if base.has_edge(a,b):base.remove_edge(a,b)
    if base.has_edge(b,a):base.remove_edge(b,a)
    variants=[]
    for label,arc in [('absent',None),('a_to_b',(a,b)),('b_to_a',(b,a))]:
        candidate=base.copy()
        if arc is not None:
            try:candidate.add_edge(*arc)
            except ValueError:continue
        if nx.is_directed_acyclic_graph(candidate):variants.append((label,bdeu.score(candidate)))
    norm=logsumexp([v for _,v in variants]);null=dict(variants)['absent']
    for label,value in variants:
        bf_rows.append(dict(a=a,b=b,state=label,log_BF_vs_absent=value-null,
            conditional_weight=np.exp(value-norm)))
display(save('arc_contributions',pd.DataFrame(contributions)),save('local_bayes_factors',pd.DataFrame(bf_rows)))

,a,b,delta_BIC,log_BF
0,サービス品質,対応速度,390.214487,392.449170
1,サービス品質,満足度,216.809444,229.462426
2,価格納得感,満足度,32.408223,47.103837
3,満足度,継続意向,303.014683,304.564151


,a,b,state,log_BF_vs_absent,conditional_weight
0,サービス品質,価格納得感,absent,0.000000,9.991262e-01
1,サービス品質,価格納得感,a_to_b,-7.734977,4.368806e-04
2,サービス品質,価格納得感,b_to_a,-7.734977,4.368806e-04
3,サービス品質,対応速度,absent,0.000000,1.821633e-171
4,サービス品質,対応速度,a_to_b,392.449170,5.000000e-01
5,サービス品質,対応速度,b_to_a,392.449170,5.000000e-01
6,サービス品質,満足度,absent,0.000000,2.216840e-100
7,サービス品質,満足度,a_to_b,229.462426,9.999998e-01
8,サービス品質,満足度,b_to_a,214.079188,2.085185e-07
9,サービス品質,継続意向,absent,0.000000,1.000000e+00


## 7. モデル平均化の類似処理
再出現率85%以上の骨格を採用候補とします。これは仮の表示閾値で、普遍的合格点ではありません。
両方向の確定頻度が同じなら未識別として記録します。確率を推定するDAGは**主モデルのトポロジカル順**で向きを補完します。頻度の高い向きと衝突する場合も表に残します。
これは閾値付き合意骨格から作る一つのDAGであり、bnlearn averaged.networkとの同一出力や予測分布のベイズモデル平均ではありません。

In [12]:
order={node:i for i,node in enumerate(nx.topological_sort(graph))}
averaged=DAG();averaged.add_nodes_from(train.columns);average_rows=[]
for r in edges[(edges.algorithm=='HC_BIC')&(edges.strength>=.85)].itertuples():
    a,b=sorted([r.a,r.b],key=lambda x:order[x]);averaged.add_edge(a,b)
    preferred=(r.a,r.b) if r.direction_ab>r.direction_ba else ((r.b,r.a) if r.direction_ba>r.direction_ab else None)
    average_rows.append(dict(a=a,b=b,strength=r.strength,orientation='reference_topological_order',
        conflicts_with_preference=preferred is not None and preferred!=(a,b)))
averaged_model=fit_parameters(averaged,train,10)
display(save('averaged_edges',pd.DataFrame(average_rows)))
print('合意DAGは有効:',averaged_model.check_model())

,a,b,strength,orientation,conflicts_with_preference
0,サービス品質,対応速度,1.000000,reference_topological_order,False
1,サービス品質,満足度,1.000000,reference_topological_order,False
2,価格納得感,満足度,0.998333,reference_topological_order,False
3,満足度,継続意向,1.000000,reference_topological_order,False


合意DAGは有効: True


## 8. CPT：件数、区間、確率差
全CPTに実際の補完後件数と固定構造・事前分布に条件づけた95%**信用区間**を付けます。これは頻度論的信頼区間と異なります。最頻値補完による不確実性はこの区間に含みません。
追加で元の不明行を再抽出・再補完し、固定構造を再推定する200回の百分位Bootstrap信頼区間を計算します。構造選択の不確実性は含みません。
確率差は各変数の最初の親だけを低→高に変え、他の親を中に固定したときの「高」の確率差です。単位は確率差（×100でポイント）で、因果効果ではありません。

CPTのBootstrap 200回は試行用です。報告時には回数を増やし（例：1,000回以上）、区間端点が十分安定するか確認してください。増やしても欠損機構や母集団の偏りは解消しません。

In [13]:
cpt=cpt_details(graph,model,train,10)
def cpt_vector(fit):
    return np.array([fit.get_cpds(r['node']).get_value(**dict(zip(r['parents'],r['context'])),
        **{r['node']:s}) for r in support_table(graph,train) for s in STATES])
def probability_contrasts(fit):
    result={}
    for node in graph.nodes():
        parents=sorted(graph.predecessors(node))
        if not parents:continue
        fixed={p:'中' for p in parents};parent=parents[0]
        cp=fit.get_cpds(node)
        result[f'{node}: {parent}高−低（他の親は中）']=cp.get_value(**{**fixed,parent:'高',node:'高'})-cp.get_value(**{**fixed,parent:'低',node:'高'})
    return result
samples=[];differences=[]
rng=np.random.default_rng(444)
for repetition in range(200):
    sampled=dev.iloc[rng.integers(0,len(dev),len(dev))]
    fit=fit_parameters(graph,preprocess_train(sampled,'mode'),10)
    samples.append(cpt_vector(fit));differences.append(probability_contrasts(fit))
cpt['bootstrap_lower95'],cpt['bootstrap_upper95']=np.quantile(samples,[.025,.975],axis=0)
save('CPT',cpt);display(cpt)
contrast=pd.DataFrame(differences)
contrast_table=pd.DataFrame({'contrast':list(probability_contrasts(model)),
    'difference':list(probability_contrasts(model).values()),
    'lower95':contrast.quantile(.025).to_numpy(),'upper95':contrast.quantile(.975).to_numpy()})
display(save('probability_contrasts',contrast_table))

,node,parents,context,state,n_context,count,probability,lower95,upper95,interval_width,prior_fraction,no_data,bootstrap_lower95,bootstrap_upper95
0,サービス品質,なし,全体,低,1200,286,0.239118,2.155099e-01,0.263544,0.048034,0.008264,False,0.215978,0.262280
1,サービス品質,なし,全体,中,1200,567,0.471350,4.432850e-01,0.499504,0.056219,0.008264,False,0.439924,0.498664
2,サービス品質,なし,全体,高,1200,347,0.289532,2.643228e-01,0.315400,0.051077,0.008264,False,0.265523,0.316825
3,価格納得感,なし,全体,低,1200,317,0.264738,2.402642e-01,0.289949,0.049685,0.008264,False,0.240751,0.284635
4,価格納得感,なし,全体,中,1200,553,0.459780,4.317779e-01,0.487907,0.056129,0.008264,False,0.433251,0.487052
5,価格納得感,なし,全体,高,1200,330,0.275482,2.506777e-01,0.300989,0.050312,0.008264,False,0.253994,0.298643
6,対応速度,サービス品質,低,低,286,218,0.757296,7.063557e-01,0.804869,0.098513,0.011521,False,0.711001,0.803806
7,対応速度,サービス品質,低,中,286,48,0.169739,1.287902e-01,0.215010,0.086219,0.011521,False,0.128478,0.210882
8,対応速度,サービス品質,低,高,286,20,0.072965,4.595569e-02,0.105551,0.059595,0.011521,False,0.043614,0.102433
9,対応速度,サービス品質,中,低,567,66,0.117670,9.255047e-02,0.145328,0.052777,0.005845,False,0.095636,0.144293


,contrast,difference,lower95,upper95
0,対応速度: サービス品質高−低（他の親は中）,0.640959,0.585809,0.702877
1,満足度: サービス品質高−低（他の親は中）,0.562339,0.483528,0.643791
2,継続意向: 満足度高−低（他の親は中）,0.753248,0.706888,0.793527


## 9. 条件付き独立性検定
主モデルが開発データで定めた仮説を、隔離した完全観測の評価行で確認します。条件内で置換を999回、同じ族の全仮説にHolm補正。
p<.05は独立性に反する材料、p≥.05は独立性の証明ではありません。小さな条件群では検出力不足があり得ます。非辺の独立性が棄却された場合はモデル不足の候補です。

In [14]:
tests=conditional_tests(test,graph_test_plan(graph),permutations=999)
display(save('conditional_tests',tests))

,kind,x,y,given,conditional_MI,p,permutations,testable,n,min_stratum_n,p_holm
0,implied_independence,サービス品質,価格納得感,なし,0.004117,0.798,999,True,209,209,1.000
1,implied_independence,価格納得感,対応速度,なし,0.001014,0.982,999,True,209,209,1.000
2,edge,対応速度,サービス品質,なし,0.409728,0.001,999,True,209,209,0.013
3,implied_independence,対応速度,価格納得感,サービス品質,0.038442,0.323,999,True,209,53,1.000
4,implied_independence,対応速度,満足度,サービス品質,0.028722,0.534,999,True,209,53,1.000
5,implied_independence,対応速度,継続意向,サービス品質,0.023023,0.816,999,True,209,53,1.000
6,edge,満足度,サービス品質,価格納得感,0.266823,0.001,999,True,209,62,0.013
7,edge,満足度,価格納得感,サービス品質,0.124719,0.001,999,True,209,53,0.013
8,implied_independence,満足度,対応速度,サービス品質 / 価格納得感,0.061584,0.691,999,True,209,17,1.000
9,implied_independence,継続意向,サービス品質,満足度,0.016540,0.578,999,True,209,52,1.000


## 10. 推論と生成：cpquery / rbnに相当
VariableEliminationで条件付き確率を厳密計算します。cpqueryのモンテカルロ近似とは計算方式が異なります。
simulateで推定モデルからサンプルを生成します。生成行は実観測でも外部検証データでもありません。

In [15]:
target='継続意向' if '継続意向' in train else train.columns[-1]
evidence_node=next(c for c in train if c!=target)
inference=VariableElimination(model)
query=inference.query([target],evidence={evidence_node:'高'},show_progress=False)
print(f'P({target} | {evidence_node}=高)');print(query)
assert np.isclose(query.values.sum(),1)
generated=model.simulate(n_samples=1000,seed=555,show_progress=False)
display(save('model_generated_data',generated).head())

P(継続意向 | サービス品質=高)
+---------+-------------+
| 継続意向    |   phi(継続意向) |
+=========+=============+
| 継続意向(低) |      0.1233 |
+---------+-------------+
| 継続意向(中) |      0.2563 |
+---------+-------------+
| 継続意向(高) |      0.6204 |
+---------+-------------+


,サービス品質,対応速度,継続意向,満足度,価格納得感
0,高,高,高,高,高
1,低,低,低,低,中
2,高,高,高,高,高
3,中,高,中,中,高
4,中,中,低,中,低


## 11. 別時点・別集団と業務知識
独立seedの同分布標本を「別時点の模擬」、満足度の高低を一部入れ替えた標本を「分布が変化した集団の模擬」として、固定モデルを評価します。実際の時期・組織の再現性を証明しません。
業務知識は担当者の確認が必要です。今回は「5項目が観測時に利用できる」「不明を未観測とみなす」という仮定のみ。利用時点・定義・逆向きの妥当性は未確認として残します。

In [16]:
external_rows=[]
if set(train.columns)==set(make_sample(10).columns):
    for label,seed,shift in [('別時点の模擬',888,False),('別集団の模擬（分布変化）',999,True)]:
        external=make_sample(1000,seed)
        if shift:
            rng=np.random.default_rng(seed);mask=rng.random(len(external))<.4
            external.loc[mask,'満足度']=external.loc[mask,'満足度'].map({'高':'低','中':'中','低':'高'})
        external=categorical(external)
        refit=learn_structure(external,'HC_BIC')
        external_rows.append(dict(scenario=label,n=len(external),nll=row_nll(model,external).mean(),
            baseline_nll=row_nll(models['Independent'],external).mean(),SHD=graph_distance(graph,refit)[0]))
external_results=save('synthetic_external',pd.DataFrame(external_rows));display(external_results)
business_review=pd.DataFrame([
    {'item':'不明は未観測か、独立した業務状態か','status':'担当者未確認'},
    {'item':'項目の定義・高中低の判定基準','status':'担当者未確認'},
    {'item':'利用時点で全説明項目を取得できるか','status':'担当者未確認'},
    {'item':'矢印を施策効果と誤解していないか','status':'説明上の注意を記載'},
    {'item':'実際の別時点・別集団で再現するか','status':'実データ未提供のため未検証'}])
display(save('business_review',business_review))

,scenario,n,nll,baseline_nll,SHD
0,別時点の模擬,1000,4.361196,5.444338,4
1,別集団の模擬（分布変化）,1000,5.739694,5.402315,5


,item,status
0,不明は未観測か、独立した業務状態か,担当者未確認
1,項目の定義・高中低の判定基準,担当者未確認
2,利用時点で全説明項目を取得できるか,担当者未確認
3,矢印を施策効果と誤解していないか,説明上の注意を記載
4,実際の別時点・別集団で再現するか,実データ未提供のため未検証


## 12. 結果の判断
「問題なし」を単一スコアで宣言せず、実行の正常性、安定性、適合不足の兆候、未確認範囲を別々にまとめます。
85%の辺頻度、95%区間幅0.20、欠損処理間NLL差0.05はこの演習の点検値で、普遍的合格点ではありません。

### bnlearnとの対応
|bnlearn|このNotebook|
|---|---|
|boot.strength|再標本化とCPDAG方向頻度。補完も毎回再実行|
|arc.strength|辺削除のΔBICと条件付き独立性検定|
|bf.strength|残りの構造固定で局所BDeu log BFとモデル重み（完全互換ではない）|
|averaged.network|閾値付き骨格＋参照順序による合意DAG（完全互換ではない）|
|ci.test|条件内置換MI＋Holm|
|score / logLik / nparams|BIC・BDeu、明示的な対数尤度、自由パラメータ数|
|bn.cv|学習fold内の補完・構造・パラメータ再学習|
|shd|追加削除反転を各1とする距離と骨格差|
|cpdag / reversible.arcs|to_pdagとundirected_edges|
|cpquery / rbn|VariableElimination.query / model.simulate|

参照：[bnlearnの辺強度](https://bnlearn.com/documentation/man/bf.strength.html)、[pgmpy推論](https://pgmpy.org/api/generated/inference/pgmpy.inference.VariableElimination.html)、[pgmpy生成](https://pgmpy.org/examples/Simulating_Data.html)。
これらの資料はAPIと手法の確認に使用。ここでの仮の点検閾値を公的な採否基準とはしていません。

In [17]:
primary_edges=edges[edges.algorithm=='HC_BIC']
violations=tests[(tests.kind=='implied_independence')&(tests.p_holm<.05)]
mode_cv=cv_summary[cv_summary.algorithm=='HC_BIC']
report=f"""# RESULT
- 実行版：{versions}
- 最終評価は完全観測の{len(test)}行に限定。
- 固定主モデルのNLL：{holdout.model_nll.iloc[0]:.4f}、独立モデル：{holdout.baseline_nll.iloc[0]:.4f}（低い方が良い）。
- NLL改善の95%評価Bootstrap区間：[{holdout.lower95.iloc[0]:.4f}, {holdout.upper95.iloc[0]:.4f}]。固定モデル・完全観測行に条件づけた評価。
- HC/BICの辺出現率85%以上：{int((primary_edges.strength>=.85).sum())}本。方向の確定性は別表を確認。
- 構造Bootstrapの失敗：{len(boot_errors)}回。成功回のみの頻度と、失敗の影響範囲を保存。
- 欠損処理別の主手法CV NLLの最大差：{mode_cv.nll.max()-mode_cv.nll.min():.4f}。小さい場合も全欠損機構への頑健性を保証しない。
- 仮定した条件付き独立性のHolm補正p<.05：{len(violations)}件。不棄却は独立性の証明ではない。
- CPTの95%信用区間幅>.20：{int((cpt.interval_width>.20).sum())}セル。補完をやり直すBootstrap区間も併記。
- マルコフ同値性、辺のスコア寄与、局所BF、確率差は各表を参照。
- 実時点・実集団での再現性と業務担当者の知識整合は未検証。人工データの模擬結果で代替しない。

**結論：検証プログラムは実行できたが、分析全体を業務上『問題なし』とは判定しない。**
未知データへの適用範囲は完全観測行に限定され、単一補完の仮定・構造の手法依存・方向の未識別を残す。
業務で必要な確率差や許容誤差を合意し、実データの外部検証後に判断する。
"""

if len(external_results):
    report += "\n## 模擬外部標本で見つかった限界\n"
    for row in external_results.itertuples():
        report += f"- {row.scenario}：主モデルNLL={row.nll:.4f}、独立モデル={row.baseline_nll:.4f}。" + ("主モデルが悪化。適用先の分布変化に注意。" if row.nll > row.baseline_nll else "この模擬標本では主モデルが改善。") + "\n"

(OUT/'RESULT.md').write_text(report,encoding='utf-8');display(Markdown(report))
assert all(m.check_model() for m in models.values())
assert np.allclose(cpt.groupby(['node','context']).probability.sum(),1)
assert len(cv)==3*5*3*4
print('モデル形式・確率和・CV実行件数のチェックを通過。')

# RESULT
- 実行版：{'pgmpy': '1.1.2', 'pandas': '2.3.3', 'numpy': '2.3.5', 'scikit-learn': '1.9.0'}
- 最終評価は完全観測の209行に限定。
- 固定主モデルのNLL：4.3530、独立モデル：5.4295（低い方が良い）。
- NLL改善の95%評価Bootstrap区間：[0.9235, 1.2433]。固定モデル・完全観測行に条件づけた評価。
- HC/BICの辺出現率85%以上：4本。方向の確定性は別表を確認。
- 構造Bootstrapの失敗：0回。成功回のみの頻度と、失敗の影響範囲を保存。
- 欠損処理別の主手法CV NLLの最大差：0.0555。小さい場合も全欠損機構への頑健性を保証しない。
- 仮定した条件付き独立性のHolm補正p<.05：0件。不棄却は独立性の証明ではない。
- CPTの95%信用区間幅>.20：1セル。補完をやり直すBootstrap区間も併記。
- マルコフ同値性、辺のスコア寄与、局所BF、確率差は各表を参照。
- 実時点・実集団での再現性と業務担当者の知識整合は未検証。人工データの模擬結果で代替しない。

**結論：検証プログラムは実行できたが、分析全体を業務上『問題なし』とは判定しない。**
未知データへの適用範囲は完全観測行に限定され、単一補完の仮定・構造の手法依存・方向の未識別を残す。
業務で必要な確率差や許容誤差を合意し、実データの外部検証後に判断する。

## 模擬外部標本で見つかった限界
- 別時点の模擬：主モデルNLL=4.3612、独立モデル=5.4443。この模擬標本では主モデルが改善。
- 別集団の模擬（分布変化）：主モデルNLL=5.7397、独立モデル=5.4023。主モデルが悪化。適用先の分布変化に注意。


モデル形式・確率和・CV実行件数のチェックを通過。


# 保存時の結果（再実行後は上の動的結果を参照）

# RESULT
- 実行版：{'pgmpy': '1.1.2', 'pandas': '2.3.3', 'numpy': '2.3.5', 'scikit-learn': '1.9.0'}
- 最終評価は完全観測の209行に限定。
- 固定主モデルのNLL：4.3530、独立モデル：5.4295（低い方が良い）。
- NLL改善の95%評価Bootstrap区間：[0.9235, 1.2433]。固定モデル・完全観測行に条件づけた評価。
- HC/BICの辺出現率85%以上：4本。方向の確定性は別表を確認。
- 構造Bootstrapの失敗：0回。成功回のみの頻度と、失敗の影響範囲を保存。
- 欠損処理別の主手法CV NLLの最大差：0.0555。小さい場合も全欠損機構への頑健性を保証しない。
- 仮定した条件付き独立性のHolm補正p<.05：0件。不棄却は独立性の証明ではない。
- CPTの95%信用区間幅>.20：1セル。補完をやり直すBootstrap区間も併記。
- マルコフ同値性、辺のスコア寄与、局所BF、確率差は各表を参照。
- 実時点・実集団での再現性と業務担当者の知識整合は未検証。人工データの模擬結果で代替しない。

**結論：検証プログラムは実行できたが、分析全体を業務上『問題なし』とは判定しない。**
未知データへの適用範囲は完全観測行に限定され、単一補完の仮定・構造の手法依存・方向の未識別を残す。
業務で必要な確率差や許容誤差を合意し、実データの外部検証後に判断する。

## 模擬外部標本で見つかった限界
- 別時点の模擬：主モデルNLL=4.3612、独立モデル=5.4443。この模擬標本では主モデルが改善。
- 別集団の模擬（分布変化）：主モデルNLL=5.7397、独立モデル=5.4023。主モデルが悪化。適用先の分布変化に注意。
